# Advanced KServe Deployment with Transformer and Explainer

In this notebook, we'll deploy a fraud detection model using KServe with:
- **Custom Transformer**: Preprocesses input data
- **Custom Predictor**: Loads and runs the trained model
- **Custom Explainer**: Provides SHAP-based explanations

This notebook uses **KServe V2 Inference Protocol**.


In [1]:
import sys
import os

# Add the project root directory to Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
    print(f"Added {project_root} to Python path")


Added /Users/prashanth.chaitanya/git-workspaces/kubeflow/kserve-example to Python path


In [2]:
# Import required libraries
import requests
import json
import numpy as np
import time
from typing import Dict, List
import subprocess


## Step 1: Build Docker Images

First, we need to build Docker images for our custom components and load them into the Kind cluster.


In [23]:
# Build all Docker images
print("Building Docker images...")
build_script = os.path.join(project_root, "docker", "build-images.sh")
result = subprocess.run(["bash", build_script], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("Error:", result.stderr)


Building Docker images...
Building KServe Custom Images
Project Root: /Users/prashanth.chaitanya/git-workspaces/kubeflow/kserve-example
Docker Hub Repo: prashanthchaitanya715/kubeflowbook
Version: latest
Push to Docker Hub: true

Building Transformer...
Building fraud-detection-transformer:latest...
Pushing prashanthchaitanya715/kubeflowbook:transformer-latest to Docker Hub...
The push refers to repository [docker.io/prashanthchaitanya715/kubeflowbook]
2b573801e465: Waiting
b017e08de583: Waiting
c649c88aa374: Waiting
e150ba8b6721: Waiting
bbe567b58f4c: Waiting
e363695fcb93: Waiting
f1e71768c414: Waiting
b74aa33083cf: Waiting
761585034929: Waiting
762b5798f87d: Waiting
761585034929: Waiting
762b5798f87d: Waiting
2b573801e465: Waiting
b017e08de583: Waiting
c649c88aa374: Waiting
e150ba8b6721: Waiting
bbe567b58f4c: Waiting
e363695fcb93: Waiting
f1e71768c414: Waiting
b74aa33083cf: Waiting
b74aa33083cf: Waiting
761585034929: Waiting
762b5798f87d: Waiting
2b573801e465: Waiting
b017e08de583: W

## Step 2: Deploy InferenceService

Deploy the KServe InferenceService with transformer, predictor, and explainer components.


In [24]:
# Deploy the InferenceService
yaml_path = os.path.join(project_root, "kserve", "advanced", "01-advanced-deployment.yaml")
print("Deploying InferenceService...")
result = subprocess.run(
    ["kubectl", "apply", "-f", yaml_path],
    capture_output=True,
    text=True
)
print(result.stdout)
if result.returncode != 0:
    print("Error:", result.stderr)


Deploying InferenceService...
inferenceservice.serving.kserve.io/fraud-detection-advanced created



## Step 3: Helper Functions (V2 Protocol)

Define helper functions for making requests using KServe V2 Inference Protocol.


In [ ]:
# Helper functions - Using KServe V2 Protocol
def make_prediction(instances: List) -> Dict:
    """Make prediction request to KServe InferenceService using V2 protocol"""
    url = "http://localhost:8080/v2/models/fraud-detection-advanced/infer"
    
    # Convert instances to V2 format
    if isinstance(instances[0], list):
        batch_size = len(instances)
        num_features = len(instances[0])
    else:
        batch_size = 1
        num_features = len(instances)
       
    
    payload = {
        "inputs": [{
            "name": "input-0",
            "shape": [batch_size, num_features],
            "datatype": "FP32",
            "data": instances
        }]
    }
    
    headers = {
        "Host": "fraud-detection-advanced.kubeflow-user-example-com.example.com",
        "Content-Type": "application/json",
        "Cookie": "oauth2_proxy_kubeflow=hGbpz2HTB4wVbQ41gIW_eO-uuTLbJf_VigP5II16agWt-yCrFX4Bs_uvOQJxiC15dGsmHXRS2TI0a-Lx_xfeqODUUbUOK9qt_fAze-CTpu76rTIb_5AQsQzKqLr1PvfS1KBJMQMPHd7uVVIHva1oNwiOgqSRmdF9lgLI_MS1WdHlksOBEKfwq8QFxlX7WRKA4h1ztD6FjyQD2I9SywNiIuGwcAtOen-sacHY0oGWucxHaxavYwzSW75EeGAZ4QXYvC3hJfrCLgasFNMtj1CjLF9Ny8oEvbtu88NRD4I3XhLTHzelcGQ-tAWiQTVjekop9HGCqAgWkKA_EM4L6fgq23Kgq07i_d3rH48HKaTdy7UpfFlqC1-U_E10eASAYF26O4KRTSYoBeXDGZcBtgCg-5IWDB-qg40kPeIL1Du3vFkvIgQWdhSh3gycZb4UU5NAuul-hxooXdLylx-ctvAqN1CYhqUZy-WqvP0aUrd5e0RYoukpwHaAMLNf0G5FWRFyXZInbHMiDdUa9TnM2SlxYGcBtkBaIVxDYwfQ8KUzBAu254n3P7QtZSHKShgms40h-9CJTNR2v4_9dnM7rgGUUc_gXqNgoy2-1g9eleclYYja0MA2Csd0-iOzNHiWzSFkV-sCZoGEw4exMl3Xz9vZPymh5lU_d73dKv3ZYSyXOw3plIgHScLhud49Iom2sa9ZbDlLFzJonuzVfELtXny_Zzl35jNRgOQ2rMwjTqTvqusEtOfkoY6mDZwEL6ViWckg_1qvLOV2bxVxYHMxty3F996kW377qtHlu-6mXV97p8BKJ5c3x3JTSg9b-tmogoZut0_h9rwgikTDkmyQXqVixWPSoFEtJMWTkT80Hj03uupKKLU8QVgGSa0kVpBalIYk1uh07zN-nyjNjc0ydaJ0I0f44pN2wU9JS9RqvKv3UEElXSIJ72WTNHY_a3q616vIk2BeUfr8dh1A9y6Ieu_kFBHzy5w5zmcVMKnECPW0Ykb_StKPlrWXPAsUjMUvdcc9rOHseuXTfQPlJZqLYj93dGW-xr7S54QffXJ91SsSvdk4DfIbkyTd_FHXa3tYizalq0jySBtgmMcfmhydcXm17GYEBnvjRKU-3RRTH53HzLLAnXyjH5R-k4PhCPzvMca5__Tnq9XTDuiox1FblxekGAWMt6HPxeeAYwgwj4Jmtbg3TKj2oGvr1uREdA4NrGjdYf7MIp2Fj87B36beoa_6ARD0u3GlGAVJ-ucUYPPIloxibxrkTgBWwOUlzA0mPNSQWr7upL0aFk5rCr704WfOBSO2MtQHwbBb5EAyh6kH_YX_D_DzkB5dyXSqr9fzViNjDKHTpCKw1KP2qQLqgYQ-b2FLRf-lzMZmuIx5iL1Ay5Au-Cyw0u9huX8f5bG-7xNShSZ6GZ2_b9kRdCvuxNPrSVS1A9pxA5LFM-mPu-EG4qrMDMbFaHL1eu_R18CV1u9JCi5Yhjgd_6FYF47KimkdvdL8-8y16uDzUEVJJP7BJ2qroMssoOS3RbElRMb58sSm7aQn4vUnsrVhzIrC2P4nNxqapIM_aTm5el0hSzd60o2NQ2ZA3r3-HQDo70fQuCU-JcW8eg2uUldpaN9Hg3jhlxYODjAhNF51nY3OZvLGebNq6BN-IweEzxD1oE4GM_xL6J7P8szabNXPBvDhkr3XiEiw2hhS3i-FkdmWUWNv20CjTnver8f4I3R_xMHcAU6WjMMQE9nHxmbGiPGYw9Vfop-HOzTz7wRsTYZBsPlcJG15qgj16GkDfn1EImieK5YzxmS5LWFS7cTqCN9FkaFQdcJVkx3oW0Fr1a94vjwcuccalOeDAeVwjB_ju-kDpUjBUs72PAR7ywAdbQlJLOxdXWGVOF_dl03vwJUQ6HUISLALvHf5tP8HgBvcbPD8|1760928549|-P4K6EkN81PXVfhLeC4uVQhG8vkfpI8OzISSwx27QO0="
    }
    
    response = requests.post(url, json=payload, headers=headers)
    response.raise_for_status()
    return response.json()

def get_explanation(instances: List) -> Dict:
    """Get model explanation using SHAP - V2 protocol"""
    url = "http://localhost:8080/v1/models/fraud-detection-advanced:explain"
    
    # Convert instances to V2 format
    if isinstance(instances[0], list):
        batch_size = len(instances)
        num_features = len(instances[0])
    else:
        batch_size = 1
        num_features = len(instances)
    
    payload = {
        "inputs": [{
            "name": "input-0",
            "shape": [batch_size, num_features],
            "datatype": "FP32",
            "data": instances
        }]
    }
    
    headers = {
        "Host": "fraud-detection-advanced.kubeflow-user-example-com.example.com",
        "Content-Type": "application/json",
        "Cookie": "oauth2_proxy_kubeflow=hGbpz2HTB4wVbQ41gIW_eO-uuTLbJf_VigP5II16agWt-yCrFX4Bs_uvOQJxiC15dGsmHXRS2TI0a-Lx_xfeqODUUbUOK9qt_fAze-CTpu76rTIb_5AQsQzKqLr1PvfS1KBJMQMPHd7uVVIHva1oNwiOgqSRmdF9lgLI_MS1WdHlksOBEKfwq8QFxlX7WRKA4h1ztD6FjyQD2I9SywNiIuGwcAtOen-sacHY0oGWucxHaxavYwzSW75EeGAZ4QXYvC3hJfrCLgasFNMtj1CjLF9Ny8oEvbtu88NRD4I3XhLTHzelcGQ-tAWiQTVjekop9HGCqAgWkKA_EM4L6fgq23Kgq07i_d3rH48HKaTdy7UpfFlqC1-U_E10eASAYF26O4KRTSYoBeXDGZcBtgCg-5IWDB-qg40kPeIL1Du3vFkvIgQWdhSh3gycZb4UU5NAuul-hxooXdLylx-ctvAqN1CYhqUZy-WqvP0aUrd5e0RYoukpwHaAMLNf0G5FWRFyXZInbHMiDdUa9TnM2SlxYGcBtkBaIVxDYwfQ8KUzBAu254n3P7QtZSHKShgms40h-9CJTNR2v4_9dnM7rgGUUc_gXqNgoy2-1g9eleclYYja0MA2Csd0-iOzNHiWzSFkV-sCZoGEw4exMl3Xz9vZPymh5lU_d73dKv3ZYSyXOw3plIgHScLhud49Iom2sa9ZbDlLFzJonuzVfELtXny_Zzl35jNRgOQ2rMwjTqTvqusEtOfkoY6mDZwEL6ViWckg_1qvLOV2bxVxYHMxty3F996kW377qtHlu-6mXV97p8BKJ5c3x3JTSg9b-tmogoZut0_h9rwgikTDkmyQXqVixWPSoFEtJMWTkT80Hj03uupKKLU8QVgGSa0kVpBalIYk1uh07zN-nyjNjc0ydaJ0I0f44pN2wU9JS9RqvKv3UEElXSIJ72WTNHY_a3q616vIk2BeUfr8dh1A9y6Ieu_kFBHzy5w5zmcVMKnECPW0Ykb_StKPlrWXPAsUjMUvdcc9rOHseuXTfQPlJZqLYj93dGW-xr7S54QffXJ91SsSvdk4DfIbkyTd_FHXa3tYizalq0jySBtgmMcfmhydcXm17GYEBnvjRKU-3RRTH53HzLLAnXyjH5R-k4PhCPzvMca5__Tnq9XTDuiox1FblxekGAWMt6HPxeeAYwgwj4Jmtbg3TKj2oGvr1uREdA4NrGjdYf7MIp2Fj87B36beoa_6ARD0u3GlGAVJ-ucUYPPIloxibxrkTgBWwOUlzA0mPNSQWr7upL0aFk5rCr704WfOBSO2MtQHwbBb5EAyh6kH_YX_D_DzkB5dyXSqr9fzViNjDKHTpCKw1KP2qQLqgYQ-b2FLRf-lzMZmuIx5iL1Ay5Au-Cyw0u9huX8f5bG-7xNShSZ6GZ2_b9kRdCvuxNPrSVS1A9pxA5LFM-mPu-EG4qrMDMbFaHL1eu_R18CV1u9JCi5Yhjgd_6FYF47KimkdvdL8-8y16uDzUEVJJP7BJ2qroMssoOS3RbElRMb58sSm7aQn4vUnsrVhzIrC2P4nNxqapIM_aTm5el0hSzd60o2NQ2ZA3r3-HQDo70fQuCU-JcW8eg2uUldpaN9Hg3jhlxYODjAhNF51nY3OZvLGebNq6BN-IweEzxD1oE4GM_xL6J7P8szabNXPBvDhkr3XiEiw2hhS3i-FkdmWUWNv20CjTnver8f4I3R_xMHcAU6WjMMQE9nHxmbGiPGYw9Vfop-HOzTz7wRsTYZBsPlcJG15qgj16GkDfn1EImieK5YzxmS5LWFS7cTqCN9FkaFQdcJVkx3oW0Fr1a94vjwcuccalOeDAeVwjB_ju-kDpUjBUs72PAR7ywAdbQlJLOxdXWGVOF_dl03vwJUQ6HUISLALvHf5tP8HgBvcbPD8|1760928549|-P4K6EkN81PXVfhLeC4uVQhG8vkfpI8OzISSwx27QO0="
    }
    
    response = requests.post(url, json=payload, headers=headers)
    response.raise_for_status()
    return response.json()

print("✅ Helper functions defined (V2 Protocol)!")


✅ Helper functions defined (V2 Protocol)!


## Step 4: Test Predictions

Test the deployed model with sample fraud detection data.


In [60]:
# Generate test data (30 features for fraud detection)
test_instances = [
    # Normal transaction
    [3.2, 2.8, 4.1, 2.5, 3.0, 2.9, 3.5, 2.7, 3.8, 4.2,
     2.6, 3.1, 2.4, 3.7, 2.8, 3.3, 2.9, 4.0, 3.6, 2.7,
     3.4, 2.5, 3.9, 3.2, 2.8, 3.0, 3.7, 2.6, 3.5, 2.9],
    # Suspicious transaction
    [8.5, 9.2, 7.8, 8.9, 9.1, 8.7, 9.4, 8.3, 9.0, 8.6,
     9.3, 8.8, 9.5, 8.4, 9.2, 8.9, 9.1, 8.7, 9.3, 8.5,
     9.0, 8.6, 9.4, 8.2, 9.1, 8.8, 9.2, 8.7, 9.0, 8.9]
]

print(f"Test instances shape: {len(test_instances)} x {len(test_instances[0])}")


Test instances shape: 2 x 30


In [61]:
# Make predictions
print("Making predictions...")
try:
    response = make_prediction(test_instances)
    print("\n📊 Prediction Response:")
    print(json.dumps(response, indent=2))
except Exception as e:
    print(e)
    print("\n💡 Tip: Make sure port-forward is running:")
    print("   kubectl port-forward -n istio-system svc/istio-ingressgateway 8080:80")


Making predictions...

📊 Prediction Response:
{
  "model_name": "fraud-detection-advanced",
  "model_version": null,
  "id": "f318f800-1103-429b-85ed-f0fe8b0810fd",
  "parameters": null,
  "outputs": [
    {
      "name": "output-0",
      "shape": [
        2
      ],
      "datatype": "INT64",
      "parameters": null,
      "data": [
        0,
        0
      ]
    }
  ]
}


## Step 6: Test Explainer

Get SHAP-based explanations for the predictions.


In [62]:
# Get explanations
print("Getting explanations...")
try:
    explanation = get_explanation([test_instances[0]])  # Explain first instance
    print("\n🔍 Explanation Response:")
    print(json.dumps(explanation, indent=2))
except Exception as e:
    print(f"❌ Error: {e}")


Getting explanations...
❌ Error: 404 Client Error: Not Found for url: http://localhost:8080/v2/models/fraud-detection-advanced/explain


## Step 7: Analyze Feature Importance

Extract and visualize the most important features from SHAP explanations.


In [ ]:
# Analyze feature importance
try:
    if 'explanations' in explanation:
        exp = explanation['explanations'][0]
        feature_importance = exp.get('feature_importance', {})
        
        # Sort by absolute importance
        sorted_features = sorted(
            feature_importance.items(),
            key=lambda x: abs(x[1]),
            reverse=True
        )
        
        print("\n🎯 Top 10 Most Important Features:")
        print("-" * 50)
        for feature, importance in sorted_features[:10]:
            direction = "↑" if importance > 0 else "↓"
            print(f"{direction} {feature}: {importance:.4f}")
            
except Exception as e:
    print(f"Error analyzing explanations: {e}")


## Step 8: Batch Predictions

Test batch prediction performance with multiple instances.


In [ ]:
# Generate batch test data
batch_size = 10
batch_instances = []
for i in range(batch_size):
    # Generate random features
    instance = np.random.uniform(2.0, 5.0, 30).tolist()
    batch_instances.append(instance)

print(f"Generated {batch_size} test instances")

# Make batch predictions
try:
    start_time = time.time()
    batch_response = make_prediction(batch_instances)
    end_time = time.time()
    
    print(f"\n⚡ Batch Prediction Performance:")
    print(f"  - Batch size: {batch_size}")
    print(f"  - Time taken: {(end_time - start_time):.3f} seconds")
    print(f"  - Throughput: {batch_size / (end_time - start_time):.2f} predictions/sec")
    
except Exception as e:
    print(f"❌ Error: {e}")


## Step 9: Monitor Service Health

Check the health and status of the deployed components.


In [ ]:
# Check pod status
print("📊 Pod Status:")
result = subprocess.run(
    ["kubectl", "get", "pods", "-n", NAMESPACE, "-l", f"serving.kserve.io/inferenceservice={SERVICE_NAME}"],
    capture_output=True,
    text=True
)
print(result.stdout)


In [ ]:
# Check InferenceService details
print("\n📋 InferenceService Details:")
result = subprocess.run(
    ["kubectl", "describe", "inferenceservice", SERVICE_NAME, "-n", NAMESPACE],
    capture_output=True,
    text=True
)
# Print last 30 lines (most relevant info)
lines = result.stdout.split('\n')
print('\n'.join(lines[-30:]))
